# ADS: Attention Divergence Score — Colab Quickstart

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/djokobandjur/ads-vit-forensics/blob/main/colab_quickstart.ipynb)

This notebook reproduces the full pipeline for the paper *"Attention Divergence Score: A Forensic Metric for Characterizing Parameter-Level Attacks in Vision Transformers"* (T-IFS-26854-2026, resubmission).

**Layout:**
- Repository scripts are cloned fresh into `/content/ads-vit-forensics/`.
- Trained models live on Google Drive (reused from the [companion work](https://github.com/djokobandjur/vit-positional-adversarial); large, persistent).
- ImageNet-100 val/ directory comes from the companion work's `00_setup_imagenet.py` utility.
- CIFAR-100 is auto-downloaded by torchvision into `/content/drive/MyDrive/cifar100_cache/`.
- All generated JSON results are written to `/content/drive/MyDrive/ads_pipeline/data/`.
- Figures are written to `/content/drive/MyDrive/ads_pipeline/output/figures/`.

**Expected runtime on a single T4/A100:** the full pipeline (all experiments + figures + verification) takes roughly 6–8h. The verification step (Section 10) runs in under 30 seconds on CPU without GPU/PyTorch.

**Per-buffer attack convention:** This work uses a per-buffer independent δ_i for each replicated PE buffer (12 per block for RoPE, 12 for ALiBi), differing from the shared-δ convention used in the companion work. The choice is methodologically motivated; see Section II.E of the manuscript and Table II for side-by-side comparison.

## 1. Setup

Mount Google Drive (for trained models, datasets, and persistent JSON output), verify GPU availability, and clone the repository into `/content/`.

### 1.1 Mount Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

### 1.2 Verify GPU

All experiment scripts require a CUDA-capable GPU. The pipeline was developed on T4 and A100 instances. The verification step (Section 10) does not require a GPU.

In [ ]:
!nvidia-smi

### 1.3 Clone repository

In [ ]:
%cd /content
!git clone https://github.com/djokobandjur/ads-vit-forensics.git
%cd /content/ads-vit-forensics

### 1.4 Create output directories on Drive

In [ ]:
import os

DATA_DIR = '/content/drive/MyDrive/ads_pipeline/data'
FIG_DIR  = '/content/drive/MyDrive/ads_pipeline/output/figures'

os.makedirs(DATA_DIR, exist_ok=True)
os.makedirs(FIG_DIR, exist_ok=True)

print('Output dirs ready:')
print(' ', DATA_DIR)
print(' ', FIG_DIR)

## 2. Dataset Preparation

ImageNet-100 requires the companion work's setup utility to extract the 100 selected classes from the ILSVRC2012 val tarball. CIFAR-100 is downloaded automatically by torchvision.

### 2.1 ImageNet-100 setup (via companion work)

Clone the companion repository and run its `00_setup_imagenet.py` utility, which unpacks 100 classes (50 images per class) from the ILSVRC2012 val tar archive on Drive into `/content/imagenet100/val/`.

Expected input: `/content/drive/MyDrive/pe_experiment/imagenet/ILSVRC2012_img_val.tar`

In [ ]:
# Clone companion repo for the setup utility
%cd /content
!git clone https://github.com/djokobandjur/vit-positional-adversarial.git
!python /content/vit-positional-adversarial/00_setup_imagenet.py
%cd /content/ads-vit-forensics

### 2.2 CIFAR-100

No preparation needed: the scripts call `torchvision.datasets.CIFAR100(download=True)` internally with cache at `/content/drive/MyDrive/cifar100_cache/`.

## 3. Main ADS Experiments

PE-only PGD attack on all 24 models (4 PE × 3 seeds × 2 datasets) with ADS computation at every ε. Produces the per-layer ADS heatmaps and ADS(L4) trajectories that drive Sections 5 and 6 of the manuscript.

Output: nested JSON `{pe_type: {seed: {accuracies, ads_layer4, ads_per_layer, ...}}}` for each ε in `[0.0, 0.001, 0.005, 0.01, 0.05, 0.1, 0.15, 0.2, 0.3, 0.5, 1.0]`.

### 3.1 ImageNet-100

In [ ]:
!python /content/ads-vit-forensics/ads_experiment.py \
    --models_dir "/content/drive/MyDrive/Trained models_ImageNet100" \
    --val_dir    "/content/imagenet100/val" \
    --output_path "/content/drive/MyDrive/ads_pipeline/data/ads_results.json"

### 3.2 CIFAR-100

In [ ]:
!python /content/ads-vit-forensics/ads_experiment_cifar.py \
    --models_dir "/content/drive/MyDrive/Trained models_CIFAR100" \
    --output_path "/content/drive/MyDrive/ads_pipeline/data/ads_results_cifar100.json"

## 4. Attack-Surface Specificity

Compares four attack surfaces (PE-only, QKV-only, MLP-only, all-weights) at matched per-buffer perturbation granularity. Produces the saturation-budget asymmetry analysis (Section 6.2): PE-only attacks require 17×–200× larger budgets than weight attacks for operational compromise.

### 4.1 ImageNet-100

In [ ]:
!python /content/ads-vit-forensics/ads_specificity.py \
    --models_dir "/content/drive/MyDrive/Trained models_ImageNet100" \
    --val_dir    "/content/imagenet100/val" \
    --output_path "/content/drive/MyDrive/ads_pipeline/data/ads_specificity.json"

### 4.2 CIFAR-100

In [ ]:
!python /content/ads-vit-forensics/ads_specificity_cifar.py \
    --models_dir "/content/drive/MyDrive/Trained models_CIFAR100" \
    --output_path "/content/drive/MyDrive/ads_pipeline/data/ads_specificity_cifar.json"

## 5. Adaptive Attacker (Section 6.7)

Tests whether an attacker who knows the ADS metric can evade detection by adding an ADS-minimization regularization term to the PGD loss: `L_evasion = -L_CE + λ · ADS(δ)`. Confirms that ADS is a structural consequence of perturbation magnitude — no setting of λ yields evasion with damage. Verified with SPSA (gradient-free zeroth-order attacker) as required by Carlini et al. (2019).

In [ ]:
!python /content/ads-vit-forensics/ads_adaptive.py \
    --models_dir "/content/drive/MyDrive/Trained models_ImageNet100" \
    --val_dir    "/content/imagenet100/val" \
    --output_path "/content/drive/MyDrive/ads_pipeline/data/ads_adaptive.json"

## 6. Threshold Universality and Reference Evasion

### 6.1 Fine ε grid + interpolated threshold

Tests universality of the detection threshold (10× baseline ADS) with a finer ε grid (14 values from 0.001 to 1.0) and log-log interpolation of the actual crossing point. Statistical test (Kruskal-Wallis) confirms whether thresholds differ significantly across PE types.

In [ ]:
!python /content/ads-vit-forensics/ads_threshold_fine.py \
    --models_dir "/content/drive/MyDrive/Trained models_ImageNet100" \
    --val_dir    "/content/imagenet100/val" \
    --output_path "/content/drive/MyDrive/ads_pipeline/data/ads_threshold_fine.json"

### 6.2 Reference set evasion

Tests whether an attacker who knows the 256 reference image indices can craft a perturbation that preserves attention on the reference set (evades ADS) while degrading accuracy on the full validation set. Probes whether reference set secrecy is required for ADS detection.

In [ ]:
!python /content/ads-vit-forensics/ads_ref_evasion.py \
    --models_dir "/content/drive/MyDrive/Trained models_ImageNet100" \
    --val_dir    "/content/imagenet100/val" \
    --output_path "/content/drive/MyDrive/ads_pipeline/data/ads_ref_evasion.json"

## 7. Residual Stream Probing

Layer-wise positional probing: for each Transformer block, trains a Ridge regression probe to predict patch position (row, col) from 768-d residual stream activations. Uses image-level GroupKFold cross-validation (5 folds) to prevent data leakage from correlated patch activations.

The peak probing layer per PE type provides one axis of the three-way convergence finding (Section 7), validating that the operation-space partition (embedding-space vs. attention-space) is consistent across attention metrics, residual-stream metrics, and ADS signatures.

### 7.1 ImageNet-100

In [ ]:
!python /content/ads-vit-forensics/ads_probing_residual.py \
    --models_dir "/content/drive/MyDrive/Trained models_ImageNet100" \
    --val_dir    "/content/imagenet100/val" \
    --output_path "/content/drive/MyDrive/ads_pipeline/data/ads_probing_residual.json"

### 7.2 CIFAR-100

In [ ]:
!python /content/ads-vit-forensics/ads_probing_residual_cifar.py \
    --models_dir "/content/drive/MyDrive/Trained models_CIFAR100" \
    --output_path "/content/drive/MyDrive/ads_pipeline/data/ads_probing_residual_cifar.json"

## 8. ROC Analysis vs Benign Shifts

Tests the operational detection boundary: ROC AUC of ADS as a binary detector between adversarial PGD-PE perturbations and realistic benign image shifts (JPEG compression, Gaussian blur, additive noise). Confirms AUC ≥ 0.82 at ε ≥ 0.1 (Learned) / ε ≥ 0.2 (RoPE), establishing operational detectability under domain shift.

In [ ]:
!python /content/ads-vit-forensics/ads_roc_analysis.py \
    --models_dir "/content/drive/MyDrive/Trained models_ImageNet100" \
    --val_dir    "/content/imagenet100/val" \
    --output_path "/content/drive/MyDrive/ads_pipeline/data/ads_roc_v2.json"

## 9. PE-to-PE Comparison and Figure Generation

### 9.1 PE-to-PE comparison

Auxiliary analysis comparing ADS signatures between PE pairs (e.g. RoPE vs ALiBi attention-space behavior).

In [ ]:
!python /content/ads-vit-forensics/ads_comparison.py \
    --models_dir "/content/drive/MyDrive/Trained models_ImageNet100" \
    --val_dir    "/content/imagenet100/val" \
    --output_path "/content/drive/MyDrive/ads_pipeline/data/ads_comparison.json"

### 9.2 Generate paper figures

Produces 4 figures (matching manuscript v20 numbering) from the JSON files.

In [ ]:
!python /content/ads-vit-forensics/generate_ads_figures.py \
    --data-dir   "/content/drive/MyDrive/ads_pipeline/data" \
    --output-dir "/content/drive/MyDrive/ads_pipeline/output"

## 10. CPU-Only Verification (~30 seconds)

`reproduce.py` reads only the archived JSON files and regenerates every numerical claim, table value, and statistical test in the manuscript with PASS/FAIL tolerances. **Requires no GPU and no PyTorch** — pure numpy/scipy/pandas. This is the canonical verification step for reviewers and external readers.

Set `--data-dir` to point at the JSON output directory from earlier sections. The script writes verification tables to `output/tables/`.

In [ ]:
# Copy JSONs from Drive to local data/ so reproduce.py finds them
import shutil
from pathlib import Path

local_data = Path('/content/ads-vit-forensics/data')
local_data.mkdir(exist_ok=True)

drive_data = Path('/content/drive/MyDrive/ads_pipeline/data')
for f in drive_data.glob('*.json'):
    shutil.copy(f, local_data / f.name)

print(f'Copied {len(list(local_data.glob("*.json")))} JSON files to {local_data}')

In [ ]:
!python /content/ads-vit-forensics/reproduce.py

## Done

All experimental artifacts are now on Drive:

- **13 JSON result files** in `/content/drive/MyDrive/ads_pipeline/data/`
- **Paper figures** in `/content/drive/MyDrive/ads_pipeline/output/figures/`
- **Verification tables** in `/content/ads-vit-forensics/output/tables/`

For methodological details on the per-buffer attack convention used in this work (and its relationship to the shared-δ convention in the companion work), see Section II.E and Table II of the manuscript, and `CHANGELOG.md` in this repository.